# 03. Kalman Filter 기초

Kalman Filter는 노이즈가 있는 센서 측정과 시스템 모델을 결합해 상태를 추정한다.

예측:
$$\hat{x}_{k|k-1}=F\hat{x}_{k-1|k-1}+Bu_k$$
$$P_{k|k-1}=FP_{k-1|k-1}F^T+Q$$

업데이트:
$$K_k=P_{k|k-1}H^T(HP_{k|k-1}H^T+R)^{-1}$$
$$\hat{x}_{k|k}=\hat{x}_{k|k-1}+K_k(z_k-H\hat{x}_{k|k-1})$$

**핵심 직관:** 모델과 센서를 둘 다 믿되, 불확실성이 작은 쪽을 더 믿는다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 1D Constant Velocity 모델

상태 $x=[position, velocity]^T$.
측정은 위치만 들어온다고 가정한다.

$$F=\begin{bmatrix}1&\Delta t\\0&1\end{bmatrix}, \qquad H=\begin{bmatrix}1&0\end{bmatrix}$$

In [ ]:
np.random.seed(12)
dt = 0.1
steps = 180

t = np.arange(steps) * dt
true_pos = 0.5 * t + 0.35 * np.sin(0.8*t)
true_vel = np.gradient(true_pos, dt)
measurement_std = 0.35
z = true_pos + np.random.randn(steps) * measurement_std

F = np.array([[1.0, dt], [0.0, 1.0]])
H = np.array([[1.0, 0.0]])
Q = np.array([[0.002, 0.0], [0.0, 0.02]])
R = np.array([[measurement_std**2]])

x_hat = np.array([0.0, 0.0])
P = np.eye(2) * 2.0
I = np.eye(2)

estimates = []
uncertainty = []
kalman_gains = []

for k in range(steps):
    # predict
    x_pred = F @ x_hat
    P_pred = F @ P @ F.T + Q

    # update
    y = np.array([z[k]]) - H @ x_pred
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_hat = x_pred + (K @ y).ravel()
    P = (I - K @ H) @ P_pred

    estimates.append(x_hat.copy())
    uncertainty.append(np.sqrt(np.diag(P)))
    kalman_gains.append(K.ravel())

estimates = np.array(estimates)
uncertainty = np.array(uncertainty)
kalman_gains = np.array(kalman_gains)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes[0].plot(t, true_pos, 'k-', lw=2.5, label='true position')
axes[0].scatter(t, z, s=14, color='gray', alpha=0.45, label='noisy measurement')
axes[0].plot(t, estimates[:,0], color='#E85D24', lw=2.2, label='Kalman estimate')
axes[0].fill_between(t, estimates[:,0]-2*uncertainty[:,0], estimates[:,0]+2*uncertainty[:,0], color='#E85D24', alpha=0.16, label='±2σ')
axes[0].grid(alpha=0.25); axes[0].legend(); axes[0].set_ylabel('position')

axes[1].plot(t, true_vel, 'k-', lw=2.2, label='true velocity')
axes[1].plot(t, estimates[:,1], color='#534AB7', lw=2.2, label='estimated velocity')
axes[1].fill_between(t, estimates[:,1]-2*uncertainty[:,1], estimates[:,1]+2*uncertainty[:,1], color='#534AB7', alpha=0.16)
axes[1].grid(alpha=0.25); axes[1].legend(); axes[1].set_ylabel('velocity'); axes[1].set_xlabel('time (s)')
plt.tight_layout()
plt.savefig('assets/03_kalman_estimate.png', dpi=150, bbox_inches='tight')
plt.show()

rmse_meas = np.sqrt(np.mean((z - true_pos)**2))
rmse_kf = np.sqrt(np.mean((estimates[:,0] - true_pos)**2))
print(f'측정 RMSE: {rmse_meas:.4f}')
print(f'Kalman 추정 RMSE: {rmse_kf:.4f}')

## 2. Kalman Gain 해석

Kalman Gain이 크면 측정을 많이 믿고, 작으면 모델 예측을 더 믿는다.
초기에는 불확실성이 커서 측정을 강하게 반영하고, 시간이 지나며 gain이 안정된다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t, kalman_gains[:,0], color='#E85D24', lw=2.5, label='K position')
axes[0].plot(t, kalman_gains[:,1], color='#534AB7', lw=2.5, label='K velocity')
axes[0].set_title('Kalman Gain 변화')
axes[0].set_xlabel('time (s)')
axes[0].grid(alpha=0.25); axes[0].legend()

axes[1].plot(t, uncertainty[:,0], color='#E85D24', lw=2.5, label='position σ')
axes[1].plot(t, uncertainty[:,1], color='#534AB7', lw=2.5, label='velocity σ')
axes[1].set_title('추정 불확실성 감소')
axes[1].set_xlabel('time (s)')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
plt.savefig('assets/03_kalman_gain_uncertainty.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 센서 노이즈 R을 바꾸면?

$R$ 이 크면 센서를 덜 믿고, 추정값이 더 부드러워진다.
$R$ 이 작으면 센서를 많이 믿고, 측정 노이즈를 더 따라간다.

In [ ]:
def run_kf(R_scale):
    R_local = np.array([[measurement_std**2 * R_scale]])
    x_hat = np.array([0.0, 0.0])
    P = np.eye(2) * 2.0
    out = []
    for k in range(steps):
        x_pred = F @ x_hat
        P_pred = F @ P @ F.T + Q
        y = np.array([z[k]]) - H @ x_pred
        S = H @ P_pred @ H.T + R_local
        K = P_pred @ H.T @ np.linalg.inv(S)
        x_hat = x_pred + (K @ y).ravel()
        P = (I - K @ H) @ P_pred
        out.append(x_hat.copy())
    return np.array(out)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, true_pos, 'k-', lw=2.5, label='true')
ax.scatter(t, z, s=12, color='gray', alpha=0.35, label='measurement')
for scale, color in [(0.2, '#E85D24'), (1.0, '#534AB7'), (5.0, '#1D9E75')]:
    est = run_kf(scale)
    ax.plot(t, est[:,0], lw=2, color=color, label=f'R scale={scale}')
ax.set_title('측정 노이즈 가정 R에 따른 필터 응답')
ax.set_xlabel('time (s)'); ax.set_ylabel('position')
ax.grid(alpha=0.25); ax.legend()
plt.savefig('assets/03_kalman_R_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 의미 | 로보틱스 활용 |
|------|------|---------------|
| 예측 | 모델로 다음 상태 추정 | odometry, IMU propagation |
| 업데이트 | 센서로 예측 보정 | GPS, LiDAR, 카메라 측정 반영 |
| 공분산 P | 추정 불확실성 | 센서 fusion 가중치 결정 |
| Q/R | 모델/센서 노이즈 | 필터 튜닝 핵심 |
| Kalman Gain | 측정 반영 비율 | 불확실성 기반 가중 평균 |